# 03 — Bronze: API Payments (Full Load + Incremental)
**Day 2 | Step 2 — Databricks side**

Loads payments from VoltGrid API into Bronze Delta table in ADLS Gen2,
then creates an internal Databricks Delta table from the same data.

**Run order:**
1. Cell 1 — Configure storage auth
2. Cell 2 — Login to VoltGrid API, get token
3. Cell 3 — Detect load type (full or incremental)
4. Cell 4 — Paginate API and write each page to Bronze Delta (ADLS)
5. Cell 5 — Register external Delta table in Hive metastore
6. Cell 6 — Create internal (DBFS-backed) Delta table
7. Cell 7 — Write pipeline audit log entry
8. Cell 8 — Verify both tables

**Prerequisites:**
- `00b_connect_storage_no_mount` Cells 1–3 already ran this session
- Key Vault secrets: `voltgrid-api-base-url`, `voltgrid-username`, `voltgrid-password`
- Bronze folder `api/payments/` exists (run `01_create_folder_structure` if not)

## Cell 1 — Configure storage auth + helper

Sets up SP OAuth for ADLS Gen2 access and defines the `abfss()` path helper.
Must run before any ADLS read or write.

In [ ]:
SCOPE = "kv-ev-scope"

storage_account  = dbutils.secrets.get(scope=SCOPE, key="adls-account-name")
sp_client_id     = dbutils.secrets.get(scope=SCOPE, key="sp-client-id")
sp_client_secret = dbutils.secrets.get(scope=SCOPE, key="sp-client-secret")
sp_tenant_id     = dbutils.secrets.get(scope=SCOPE, key="sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", sp_client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", sp_client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{sp_tenant_id}/oauth2/token")

def abfss(container: str, path: str = "") -> str:
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return f"{base}/{path}" if path else base

print(f"Storage account : {storage_account}")
print("Storage OAuth configured — OK")

## Cell 2 — Login to VoltGrid API and get token

Sends username + password to `POST /api/auth/login/` and gets a short-lived token.
Token is stored in `API_HEADERS` — used in every subsequent API call.
The token is never written to disk or storage — only lives in notebook memory.

In [ ]:
import requests

api_base_url = dbutils.secrets.get(scope=SCOPE, key="voltgrid-api-base-url")
username     = dbutils.secrets.get(scope=SCOPE, key="voltgrid-username")
password     = dbutils.secrets.get(scope=SCOPE, key="voltgrid-password")

resp = requests.post(
    f"{api_base_url}/api/auth/login/",
    json={"username": username, "password": password},
    timeout=10,
)
resp.raise_for_status()
token = resp.json()["token"]

API_HEADERS = {"Authorization": f"Token {token}"}

print(f"API base URL : {api_base_url}")
print(f"Token        : {token[:8]}...[REDACTED]")
print("API login — OK")

## Cell 3 — Detect load type (full or incremental)

Checks whether a prior run has written data to the Bronze Delta path.

- If the path has no Delta files → **full load** (no `updated_after` filter)
- If the path already has data → **incremental load** (pass `updated_after=max(updated_at)` from last run)

`max(updated_at)` from the existing Delta data = the high-watermark.
The API returns only rows changed after that timestamp.

In [ ]:
from pyspark.sql.functions import max as spark_max

BRONZE_PAYMENTS_PATH = abfss("bronze", "api/payments/")

try:
    existing_df = spark.read.format("delta").load(BRONZE_PAYMENTS_PATH)
    row_count   = existing_df.count()

    if row_count == 0:
        LOAD_TYPE    = "full"
        UPDATED_AFTER = ""
    else:
        LOAD_TYPE     = "incremental"
        UPDATED_AFTER = existing_df.select(spark_max("updated_at")).collect()[0][0]

except Exception:
    LOAD_TYPE     = "full"
    UPDATED_AFTER = ""

print(f"Load type     : {LOAD_TYPE}")
print(f"Updated after : {UPDATED_AFTER if UPDATED_AFTER else '(none — full load)'}")

## Cell 4 — Paginate API and write each page to Bronze Delta

Loops through all pages of the payments endpoint.

**Full load:** calls `GET /api/db/payments/?page=N&page_size=100` — no date filter.
**Incremental:** adds `&updated_after={UPDATED_AFTER}` — API returns only changed rows.

Each page of 100 records is written as an append to the Bronze Delta table.
Delta handles file layout — you do not manage Parquet files manually.

`rows_written` tracks total rows loaded this run for the audit log.

In [ ]:
import json
from pyspark.sql import Row
from datetime import datetime, timezone

PAGE_SIZE    = 100
page         = 1
total_pages  = 1
rows_written = 0
run_start    = datetime.now(timezone.utc).isoformat()

print(f"Starting {LOAD_TYPE} load — {run_start}")
print(f"{'Page':>6}  {'Rows this page':>15}  {'Total rows':>12}")
print("-" * 40)

while page <= total_pages:
    url = f"{api_base_url}/api/db/payments/?page={page}&page_size={PAGE_SIZE}"
    if UPDATED_AFTER:
        url += f"&updated_after={UPDATED_AFTER}"

    r = requests.get(url, headers=API_HEADERS, timeout=30)
    r.raise_for_status()
    body = r.json()

    records    = body.get("data", [])
    pagination = body.get("pagination", {})
    total_pages = pagination.get("total_pages", 1)

    if not records:
        print(f"  Page {page}: no records returned — stopping.")
        break

    df_page = spark.createDataFrame([Row(**rec) for rec in records])

    df_page.write \
        .format("delta") \
        .mode("append") \
        .save(BRONZE_PAYMENTS_PATH)

    rows_written += len(records)
    print(f"  {page:>4}   {len(records):>14,}   {rows_written:>12,}")
    page += 1

print("-" * 40)
print(f"Load complete. Total rows written: {rows_written:,}")

## Cell 5 — Register external Delta table in Hive metastore

Creates `bronze.payments` as an **external** table — the schema is registered in Hive
but the data files remain in ADLS Gen2.

After this, you can query the table with `SELECT * FROM bronze.payments` in any notebook
or in the Databricks SQL editor without specifying the ADLS path each time.

`IF NOT EXISTS` means re-running this cell is safe — it will not overwrite the table.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze COMMENT 'Bronze layer — raw ingested data'")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.payments
    USING DELTA
    LOCATION '{BRONZE_PAYMENTS_PATH}'
    COMMENT 'Bronze payments — raw from VoltGrid API, full+incremental load'
""")

count = spark.sql("SELECT COUNT(*) FROM bronze.payments").collect()[0][0]
print(f"External table bronze.payments — OK")
print(f"Row count : {count:,}")
print(f"Location  : {BRONZE_PAYMENTS_PATH}")

## Cell 6 — Create internal (DBFS-backed) Delta table

Reads the same data from ADLS and writes it into a **managed** Databricks Delta table.
Data lives in DBFS at `/user/hive/warehouse/bronze.db/payments_internal/`.

This is for learning only — compare it to the external table and understand the difference.
From Day 3 onwards, use external tables only.

In [ ]:
df_adls = spark.read.format("delta").load(BRONZE_PAYMENTS_PATH)

df_adls.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.payments_internal")

int_count = spark.sql("SELECT COUNT(*) FROM bronze.payments_internal").collect()[0][0]
ext_count = spark.sql("SELECT COUNT(*) FROM bronze.payments").collect()[0][0]

print(f"Internal table bronze.payments_internal — OK")
print(f"External rows : {ext_count:,}")
print(f"Internal rows : {int_count:,}")
print(f"Match         : {ext_count == int_count}")

## Cell 7 — Write pipeline audit log

Writes one row to `bronze/api/pipeline_audit/` recording what this run did.
The `watermark_value` field stores `max(updated_at)` from the data just loaded.
Next run reads this to know where to resume from.

This is the watermark store — the foundation of the incremental load pattern.

In [ ]:
import uuid
from pyspark.sql.functions import max as spark_max

run_end      = datetime.now(timezone.utc).isoformat()
df_loaded    = spark.read.format("delta").load(BRONZE_PAYMENTS_PATH)
new_watermark = df_loaded.select(spark_max("updated_at")).collect()[0][0]

audit_row = Row(
    run_id         = str(uuid.uuid4()),
    pipeline_name  = "bronze_api_payments",
    source_name    = "voltgrid_api",
    load_type      = LOAD_TYPE,
    status         = "SUCCESS",
    rows_read      = rows_written,
    rows_written   = rows_written,
    rows_rejected  = 0,
    started_at     = run_start,
    ended_at       = run_end,
    error_message  = None,
    watermark_value = str(new_watermark),
)

df_audit = spark.createDataFrame([audit_row])

df_audit.write \
    .format("delta") \
    .mode("append") \
    .save(abfss("bronze", "api/pipeline_audit/"))

print(f"Audit log written — OK")
print(f"New watermark : {new_watermark}")
print(f"Next run will use: updated_after={new_watermark}")

## Cell 8 — Verify both tables

Final check: both tables have data, Delta log exists, and a sample of records looks correct.

In [ ]:
print("=== Verification ===")

ext = spark.sql("SELECT COUNT(*) FROM bronze.payments").collect()[0][0]
int_ = spark.sql("SELECT COUNT(*) FROM bronze.payments_internal").collect()[0][0]
print(f"  bronze.payments (external)  : {ext:,} rows")
print(f"  bronze.payments_internal    : {int_:,} rows")

print("\nSchema:")
spark.sql("DESCRIBE bronze.payments").show(truncate=False)

print("Sample records (external):")
display(spark.sql("SELECT payment_id, amount_aud, status, updated_at FROM bronze.payments LIMIT 5"))

print("Delta history:")
spark.sql("DESCRIBE HISTORY bronze.payments").select(
    "version", "timestamp", "operation", "operationParameters"
).show(5, truncate=False)